## 03 — Multi-day more-metrics (≥ 1 foot standing water)

This notebook computes confusion-matrix metrics for the **image-level** task:

- **Positive class**: the image shows **more than a foot of standing water** (question `q1`).
- **Prediction**: derived by factorizing raw VLM output text (`response_1`) into **yes/no**.
- **Ground truth**: a day-specific annotated set.

Key refactor: for Sep 29 (and other days), we **load chunked raw model outputs** from `notebooks/cambrian/*_{0..5}.csv` and factorize them, instead of relying on pre-aggregated/derived artifacts.

Days included:

- Sep 29 (NYC)
- California
- Jan 10
- Dec 18

Metrics reported:

- Positive predictive value (**PPV / precision**)
- **False omission rate (FOR)**
- **Recall**
- **F1-score**
- **Critical success index (CSI / threat score)**


In [1]:
from __future__ import annotations

from pathlib import Path
import re

import numpy as np
import pandas as pd

# Reproducibility
RANDOM_SEED = 777
rng = np.random.default_rng(RANDOM_SEED)

BASE_DIR = Path("../../")
CAMBRIAN_DIR = BASE_DIR / "notebooks" / "cambrian"

# -------------------------
# Inputs / configuration
# -------------------------

# Sep29 ground-truth labels (contains `image` + numeric `gt`)
SEP29_ANNOTATED_PATH = BASE_DIR / "data" / "processed" / "inspection_set.csv"

# Other-day ground-truth labels (Label Studio exports; contains `image` + string `choice`)
ANNOTATED_SAMPLE_PATHS = {
    "california": CAMBRIAN_DIR / "california_sample_annotated.csv",
    "jan10": CAMBRIAN_DIR / "jan10_sample_annotated.csv",
    "dec18": CAMBRIAN_DIR / "dec18_sample_annotated.csv",
}

# Chunked raw model outputs (free-text `response_1` to factorize)
# Sep29 has two directories; set this to "default" or "13b" depending on which chunks you want.
SEP29_VARIANT = "13b"  # {"default", "13b"}

# Population counts (Total N and Predicted Positive counts) for adjustment.
# These were calculated by scanning the entire dataset for each day.
POPULATION_STATS = {
    "sep29": {"N": 926212, "pos": 1465},     # from notebooks/cambrian/entire_sep29_all.csv
    "california": {"N": 24006, "pos": 7},
    "jan10": {"N": 160206, "pos": 16},
    "dec18": {"N": 198606, "pos": 94},
}


def sep29_chunk_paths() -> list[Path]:
    base = CAMBRIAN_DIR if SEP29_VARIANT == "default" else (CAMBRIAN_DIR / "13b")
    return sorted(base.glob("entire_sep29_[0-9].csv"))


CHUNK_PATHS = {
    "california": sorted(CAMBRIAN_DIR.glob("california_[0-9].csv")),
    "jan10": sorted(CAMBRIAN_DIR.glob("jan10_[0-9].csv")),
    "dec18": sorted(CAMBRIAN_DIR.glob("dec18_[0-9].csv")),
}


def safe_div(num: float, den: float) -> float:
    return float(num) / float(den) if den != 0 else float("nan")


CAMBRIAN_DIR


PosixPath('../../notebooks/cambrian')

In [2]:
import constants as c
from pathlib import Path
PAPER_PATH = Path(c.PAPER_PATH)

In [3]:
# Helpers: parse image IDs, factorize yes/no from response_1, and load chunk predictions

YES_RE = re.compile(r"^\s*[\"']?\s*yes\b", flags=re.IGNORECASE)
NO_RE = re.compile(r"^\s*[\"']?\s*no\b", flags=re.IGNORECASE)


def image_name_from_any(path_or_filename: str) -> str:
    """Normalize various path formats down to the basename filename."""
    s = str(path_or_filename)
    if "?d=" in s:
        s = s.split("?d=", 1)[1]
    return Path(s).name


def factorize_yes_no(text: str) -> int | pd._libs.missing.NAType:
    """Return 1 for Yes, 0 for No, NA if unknown/ambiguous."""
    if pd.isna(text):
        return pd.NA
    s = str(text).strip()
    if YES_RE.match(s):
        return 1
    if NO_RE.match(s):
        return 0
    return pd.NA


def load_chunk_preds(paths: list[Path], wanted_image_names: set[str], chunksize: int = 200_000) -> pd.DataFrame:
    """Stream CSV chunks and keep predictions only for the labeled images we care about."""
    if len(paths) == 0:
        raise FileNotFoundError("No chunk files found.")

    kept = []
    for p in paths:
        for chunk in pd.read_csv(p, chunksize=chunksize):
            # Identify image path column
            if "image_path" in chunk.columns:
                col = "image_path"
            elif "img_path" in chunk.columns:
                col = "img_path"
                raise KeyError(f"{p} missing an image path column (expected image_path or img_path)")

            if "response_1" not in chunk.columns:
                raise KeyError(f"{p} missing response_1")

            sub = chunk[[col, "response_1"]].copy()
            sub["image_name"] = sub[col].map(image_name_from_any)
            sub = sub[sub["image_name"].isin(wanted_image_names)]
            if len(sub) == 0:
                continue

            sub["pred_bin"] = sub["response_1"].map(factorize_yes_no)
            kept.append(sub[["image_name", "pred_bin"]])

    if len(kept) == 0:
        return pd.DataFrame({"image_name": [], "pred_bin": []})

    out = pd.concat(kept, ignore_index=True)
    # De-dup just in case (keep first observed)
    out = out.drop_duplicates(subset=["image_name"], keep="first")
    return out


# quick smoke checks
sep29_chunk_paths()[:2], {k: len(v) for k, v in CHUNK_PATHS.items()}


([PosixPath('../../notebooks/cambrian/13b/entire_sep29_0.csv'),
  PosixPath('../../notebooks/cambrian/13b/entire_sep29_1.csv')],
 {'california': 6, 'jan10': 6, 'dec18': 6})

In [4]:
# Load each day's ground truth, join against chunked predictions, and compute metrics

def confusion_and_metrics_weighted(tp, fp, tn, fn, w1, w0) -> dict[str, float]:
    """Compute population-adjusted metrics using stratified sampling weights."""
    tp_pop = tp * w1
    fp_pop = fp * w1
    tn_pop = tn * w0
    fn_pop = fn * w0
    n_pop = tp_pop + fp_pop + tn_pop + fn_pop

    return {
        "n_eval_sample": int(tp + fp + tn + fn),
        "n_pos_sample": int(tp + fn),
        "n_eval": int(n_pop),
        "n_pos": int(tp_pop + fn_pop),
        "n_pred_pos": int(tp_pop + fp_pop),
        "n_pred_neg": int(tn_pop + fn_pop),
        "prevalence_gt": safe_div(tp_pop + fn_pop, n_pop),
        "predicted_positive_rate": safe_div(tp_pop + fp_pop, n_pop),
        "ppv_precision": safe_div(tp_pop, tp_pop + fp_pop),
        "false_omission_rate": safe_div(fn_pop, fn_pop + tn_pop),
        "recall": safe_div(tp_pop, tp_pop + fn_pop),
        "f1": safe_div(2 * tp_pop, 2 * tp_pop + fp_pop + fn_pop),
        "csi_threat_score": safe_div(tp_pop, tp_pop + fp_pop + fn_pop),
    }


def eval_day_weighted(day, gt_df, paths):
    wanted = set(gt_df["image_name"].unique())
    preds = load_chunk_preds(paths, wanted)
    merged = gt_df.merge(preds, on="image_name", how="left")
    
    n_gt = len(gt_df)
    n_found = int(merged["pred_bin"].notna().sum())
    diagnostics.append({"day": day, "n_gt": n_gt, "n_pred_found": n_found, "coverage": n_found / n_gt if n_gt else float("nan")})

    work = merged.dropna(subset=["pred_bin"]).copy()
    work["pred_bin"] = work["pred_bin"].astype(int)
    
    tp = int(((work["gt_bin"] == 1) & (work["pred_bin"] == 1)).sum())
    fp = int(((work["gt_bin"] == 0) & (work["pred_bin"] == 1)).sum())
    tn = int(((work["gt_bin"] == 0) & (work["pred_bin"] == 0)).sum())
    fn = int(((work["gt_bin"] == 1) & (work["pred_bin"] == 0)).sum())

    pop_n = POPULATION_STATS[day]["N"]
    pop_pos = POPULATION_STATS[day]["pos"]
    pop_neg = pop_n - pop_pos

    n_pos_pred = tp + fp
    n_neg_pred = tn + fn

    w1 = pop_pos / n_pos_pred if n_pos_pred > 0 else 0
    w0 = pop_neg / n_neg_pred if n_neg_pred > 0 else 0
    
    res = {"day": day, "variant": SEP29_VARIANT if day == "sep29" else "default"}
    res |= confusion_and_metrics_weighted(tp, fp, tn, fn, w1, w0)
    return res, merged


def load_sep29_gt() -> pd.DataFrame:
    df = pd.read_csv(SEP29_ANNOTATED_PATH)
    if "image" not in df.columns or "response_1" not in df.columns:
        raise KeyError("Expected columns `image` and `gt` in inspection_set.csv")
    out = df[["image", "choice"]].copy()
    out["image_name"] = out["image"].map(image_name_from_any)

    # gt column = 1 if 'Yes' in response_1, 0 otherwise
    out["gt_bin"] = (out["choice"].astype(str) == "Flooded road").astype(int)

    out = out.dropna(subset=["gt_bin"]).copy()
    out["gt_bin"] = out["gt_bin"].astype(int)
    return out[["image_name", "gt_bin"]]


def load_otherday_gt(day: str, path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    if "image" not in df.columns or "choice" not in df.columns:
        raise KeyError(f"Expected columns `image` and `choice` in {path}")
    out = df[["image", "choice"]].copy()
    out["image_name"] = out["image"].map(image_name_from_any)
    out["gt_bin"] = (out["choice"].astype(str) == "Flooded road").astype(int)
    return out[["image_name", "gt_bin"]]


results = []
diagnostics = []

# Sep29
sep29_gt = load_sep29_gt()
res, sep29_merged = eval_day_weighted("sep29", sep29_gt, sep29_chunk_paths())
results.append(res)

# Other days
for day, gt_path in ANNOTATED_SAMPLE_PATHS.items():
    gt = load_otherday_gt(day, gt_path)
    res, _ = eval_day_weighted(day, gt, CHUNK_PATHS[day])
    results.append(res)

pd.DataFrame(results).set_index("day").sort_index()


,variant,n_eval_sample,n_pos_sample,n_eval,n_pos,n_pred_pos,n_pred_neg,prevalence_gt,predicted_positive_rate,ppv_precision,false_omission_rate,recall,f1,csi_threat_score
day,,,,,,,,,,,,,,
california,default,257,1,24006,1,7,23999,0.000042,0.000292,0.142857,0.000,1.00000,0.250000,0.142857
dec18,default,344,66,198606,66,94,198512,0.000332,0.000473,0.702128,0.000,1.00000,0.825000,0.702128
jan10,default,266,13,160206,13,16,160190,0.000081,0.000100,0.812500,0.000,1.00000,0.896552,0.812500
sep29,13b,1000,332,926211,6512,1465,924746,0.007031,0.001582,0.658000,0.006,0.14802,0.241674,0.137445


In [5]:
# Diagnostics: how many labeled images were found in the chunk predictions?

pd.DataFrame(diagnostics).set_index("day").sort_index()


,n_gt,n_pred_found,coverage
day,,,
california,257,257,1.0
dec18,344,344,1.0
jan10,266,266,1.0
sep29,1000,1000,1.0


In [6]:
# Optional: bootstrap SD (per day) over population resampling

# pd max columns display to 50 
pd.set_option('display.max_columns', 50)

DO_BOOTSTRAP = True
N_BOOT = 100000


def bootstrap_stats(y_true: np.ndarray, y_pred: np.ndarray, pop_n: int, pop_pos: int, n_boot: int) -> dict[str, float]:
    pos_idx = np.where(y_pred == 1)[0]
    neg_idx = np.where(y_pred == 0)[0]
    
    n1_sample = len(pos_idx)
    n0_sample = len(neg_idx)
    
    # Observed stratum success rates (P(y=1 | y_hat))
    p1_obs = (y_true[pos_idx] == 1).mean() if n1_sample > 0 else 0
    p0_obs = (y_true[neg_idx] == 1).mean() if n0_sample > 0 else 0
    
    # Population counts (scanned from total dataset)
    n1_pop = pop_pos
    n0_pop = pop_n - n1_pop
    
    keys = ["ppv_precision", "false_omission_rate", "recall", "f1", "csi_threat_score", "prevalence_gt", "predicted_positive_rate", "n_pred_pos", "n_pred_neg"]
    boot = {k: [] for k in keys}

    for _ in range(n_boot):
        # Sample conditional probabilities from the annotation binomials
        # (Equivalent to resampling the ground truth pool with replacement)
        p1 = rng.binomial(n1_sample, p1_obs) / n1_sample if n1_sample > 0 else 0
        p0 = rng.binomial(n0_sample, p0_obs) / n0_sample if n0_sample > 0 else 0
        
        # Scale sampled probabilities to population counts
        tp = n1_pop * p1
        fp = n1_pop * (1 - p1)
        fn = n0_pop * p0
        tn = n0_pop * (1 - p0)
        
        # Compute metrics on these population-scale counts
        m = confusion_and_metrics_weighted(tp, fp, tn, fn, 1.0, 1.0)
        for k in keys:
            boot[k].append(m[k])

    out = {}
    for k in keys:
        arr = np.asarray(boot[k], dtype=float)
        out[f"{k}_sd"] = float(np.nanstd(arr))
        out[f"{k}_ci_lower"] = float(np.nanquantile(arr, 0.025))
        out[f"{k}_ci_upper"] = float(np.nanquantile(arr, 0.975))
    return out


# Re-seed so this cell is reproducible regardless of prior rng use.
rng = np.random.default_rng(RANDOM_SEED)

if DO_BOOTSTRAP:
    rows = []
    for day in ["sep29", "california", "jan10", "dec18"]:
        if day == "sep29":
            gt = load_sep29_gt()
            paths = sep29_chunk_paths()
        else:
            gt = load_otherday_gt(day, ANNOTATED_SAMPLE_PATHS[day])
            paths = CHUNK_PATHS[day]
            
        wanted = set(gt["image_name"].unique())
        preds = load_chunk_preds(paths, wanted)
        merged = gt.merge(preds, on="image_name", how="left").dropna(subset=["pred_bin"]).copy()
        merged["pred_bin"] = merged["pred_bin"].astype(int)
        
        y_true = merged["gt_bin"].to_numpy(dtype=int)
        y_pred = merged["pred_bin"].to_numpy(dtype=int)
        
        pop_stats = POPULATION_STATS[day]
        
        tp = int(((y_true == 1) & (y_pred == 1)).sum())
        fp = int(((y_true == 0) & (y_pred == 1)).sum())
        tn = int(((y_true == 0) & (y_pred == 0)).sum())
        fn = int(((y_true == 1) & (y_pred == 0)).sum())
        
        w1 = pop_stats["pos"] / (tp + fp) if (tp + fp) > 0 else 0
        w0 = (pop_stats["N"] - pop_stats["pos"]) / (tn + fn) if (tn + fn) > 0 else 0
        
        base = {"day": day}
        base |= confusion_and_metrics_weighted(tp, fp, tn, fn, w1, w0)
        base |= bootstrap_stats(y_true, y_pred, pop_stats["N"], pop_stats["pos"], N_BOOT)
        rows.append(base)

    display(pd.DataFrame(rows).set_index("day").sort_index())
else:
    print("Set DO_BOOTSTRAP=True to compute per-day bootstrap confidence intervals.")


,n_eval_sample,n_pos_sample,n_eval,n_pos,n_pred_pos,n_pred_neg,prevalence_gt,predicted_positive_rate,ppv_precision,false_omission_rate,recall,f1,csi_threat_score,ppv_precision_sd,ppv_precision_ci_lower,ppv_precision_ci_upper,false_omission_rate_sd,false_omission_rate_ci_lower,false_omission_rate_ci_upper,recall_sd,recall_ci_lower,recall_ci_upper,f1_sd,f1_ci_lower,f1_ci_upper,csi_threat_score_sd,csi_threat_score_ci_lower,csi_threat_score_ci_upper,prevalence_gt_sd,prevalence_gt_ci_lower,prevalence_gt_ci_upper,predicted_positive_rate_sd,predicted_positive_rate_ci_lower,predicted_positive_rate_ci_upper,n_pred_pos_sd,n_pred_pos_ci_lower,n_pred_pos_ci_upper,n_pred_neg_sd,n_pred_neg_ci_lower,n_pred_neg_ci_upper
day,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
california,257,1,24006,1,7,23999,0.000042,0.000292,0.142857,0.000,1.00000,0.250000,0.142857,0.132029,0.000000,0.428571,0.000000,0.0,0.000,0.000000,1.000000,1.0,0.193151,0.000000,0.600000,0.132029,0.000000,0.428571,0.000038,0.000000,0.000125,5.421011e-20,0.000292,0.000292,0.0,7.0,7.0,0.000000,23999.0,23999.0
dec18,344,66,198606,66,94,198512,0.000332,0.000473,0.702128,0.000,1.00000,0.825000,0.702128,0.047112,0.606383,0.787234,0.000000,0.0,0.000,0.000000,1.000000,1.0,0.032714,0.754967,0.880952,0.047112,0.606383,0.787234,0.000022,0.000287,0.000373,0.000000e+00,0.000473,0.000473,0.0,94.0,94.0,0.000000,198512.0,198512.0
jan10,266,13,160206,13,16,160190,0.000081,0.000100,0.812500,0.000,1.00000,0.896552,0.812500,0.097690,0.625000,1.000000,0.000000,0.0,0.000,0.000000,1.000000,1.0,0.061445,0.769231,1.000000,0.097690,0.625000,1.000000,0.000010,0.000062,0.000100,0.000000e+00,0.000100,0.000100,0.0,16.0,16.0,0.000000,160190.0,160190.0
sep29,1000,332,926211,6512,1465,924746,0.007031,0.001582,0.658000,0.006,0.14802,0.241674,0.137445,0.021208,0.616000,0.700000,0.003448,0.0,0.014,0.196707,0.069886,1.0,0.148913,0.126462,0.793727,0.127411,0.067499,0.658000,0.003443,0.001041,0.015006,4.335296e-19,0.001582,0.001582,0.0,1465.0,1465.0,0.030482,924747.0,924747.0


### M6 — reproduce the SI headline numbers

The Supplementary "VLM performance" section quotes two sets of numbers that were
not previously produced by any notebook:

1. **"images classified flooded are at least 110 / 351 / 406 / 72 times likelier
   to be flooded"** (main text + SI) — the per-day likelihood ratio
   $\mathrm{LR}=p(y{=}1\mid\hat y{=}1)/p(y{=}1\mid\hat y{=}0)=\mathrm{PPV}/\mathrm{FOR}$.
2. The **primary-dataset 95% CIs**: PPV 0.62–0.70, FOR 0.00–0.01, recall
   0.07–1.00, F1 0.13–0.79.

The cell below computes both from the per-day confusion counts. **Note on the
ratio bound:** the three validation days have *zero* observed false negatives, so
FOR must be upper-bounded. The committed text (351/406/72) corresponds to a fixed
$\mathrm{FOR_{ub}}=1/500$ (PPV$\times$500, the nominal 250-pos $+$ 250-neg sample
size). The data-faithful conservative bound — one false negative over the
*actual* annotated predicted-negatives for that day — gives roughly half
(176/204/36). Both are printed; see the flag in the output.

In [7]:
# M6: per-day likelihood ratios + primary-dataset 95% CIs (reproduces SI numbers).
# Reuses load_sep29_gt / load_otherday_gt / load_chunk_preds / *_chunk_paths from above.

def _confusion(day):
    if day == "sep29":
        gt, paths = load_sep29_gt(), sep29_chunk_paths()
    else:
        gt, paths = load_otherday_gt(day, ANNOTATED_SAMPLE_PATHS[day]), CHUNK_PATHS[day]
    m = gt.merge(load_chunk_preds(paths, set(gt["image_name"].unique())),
                 on="image_name", how="left").dropna(subset=["pred_bin"])
    yt, yp = m["gt_bin"].to_numpy(int), m["pred_bin"].astype(int).to_numpy()
    tp = int(((yt == 1) & (yp == 1)).sum()); fp = int(((yt == 0) & (yp == 1)).sum())
    tn = int(((yt == 0) & (yp == 0)).sum()); fn = int(((yt == 1) & (yp == 0)).sum())
    return tp, fp, tn, fn

DAY_ORDER = ["sep29", "dec18", "jan10", "california"]
NOMINAL_NEG_SAMPLE = 500  # committed-text bound: 1 FN over the nominal 250+250 design

print("Per-day flooded-classification likelihood ratio  LR = PPV / FOR")
print(f"{'day':11} {'PPV':>6} {'FOR':>8} {'LR(data-faithful)':>18} {'LR(committed,1/500)':>20}")
for day in DAY_ORDER:
    tp, fp, tn, fn = _confusion(day)
    ppv = tp / (tp + fp) if tp + fp else float("nan")
    n_pred_neg = tn + fn
    if fn > 0:                                   # sep29: real false negatives observed
        forr = fn / n_pred_neg
        lr_faithful = ppv / forr
        lr_committed = lr_faithful               # no bound needed
    else:                                        # validation days: 0 FN -> upper-bound FOR
        forr = 0.0
        lr_faithful = ppv / (1.0 / (n_pred_neg + 1))    # 1 FN over actual pred-negatives
        lr_committed = ppv * NOMINAL_NEG_SAMPLE         # 1 FN over nominal 500
    print(f"{day:11} {ppv:6.3f} {forr:8.4f} {lr_faithful:18.0f} {lr_committed:20.0f}")

print("\n  committed text: 'at least 351, 406, 72' (dec18/jan10/california) <- the 1/500 column")
print("  data-faithful : 'at least 176, 204, 36'                          <- the 1/(n_pred_neg+1) column")
print("  (sep29 = 110 either way; it has observed false negatives.)")

# ---- primary-dataset (sep29) 95% CIs: PPV, FOR, and Bayes-rule recall / F1 ----
import numpy as np
_rng = np.random.default_rng(RANDOM_SEED)
tp, fp, tn, fn = _confusion("sep29")
npp, npn = tp + fp, tn + fn
ppv_hat, for_hat = tp / npp, fn / npn
p1 = POPULATION_STATS["sep29"]["pos"] / POPULATION_STATS["sep29"]["N"]; p0 = 1 - p1
B = 200_000
s_ppv = _rng.binomial(npp, ppv_hat, B) / npp
s_for = _rng.binomial(npn, for_hat, B) / npn
s_recall = (s_ppv * p1) / (s_ppv * p1 + s_for * p0)          # Bayes' rule (see SI)
s_f1 = 2 * s_ppv * s_recall / (s_ppv + s_recall)
def _ci(a): return float(np.nanpercentile(a, 2.5)), float(np.nanpercentile(a, 97.5))
print("\nPrimary dataset (9/29/23) 95% bootstrap CIs:")
for name, arr, paper in [("PPV", s_ppv, "0.62-0.70"), ("FOR", s_for, "0.00-0.01"),
                          ("recall", s_recall, "0.07-1.00"), ("F1", s_f1, "0.13-0.79")]:
    lo, hi = _ci(arr)
    print(f"  {name:7} {lo:.2f} - {hi:.2f}   (SI: {paper})")

Per-day flooded-classification likelihood ratio  LR = PPV / FOR
day            PPV      FOR  LR(data-faithful)  LR(committed,1/500)


sep29        0.658   0.0060                110                  110


dec18        0.702   0.0000                176                  351


jan10        0.812   0.0000                204                  406
california   0.143   0.0000                 36                   71

  committed text: 'at least 351, 406, 72' (dec18/jan10/california) <- the 1/500 column
  data-faithful : 'at least 176, 204, 36'                          <- the 1/(n_pred_neg+1) column
  (sep29 = 110 either way; it has observed false negatives.)



Primary dataset (9/29/23) 95% bootstrap CIs:
  PPV     0.62 - 0.70   (SI: 0.62-0.70)
  FOR     0.00 - 0.01   (SI: 0.00-0.01)
  recall  0.07 - 1.00   (SI: 0.07-1.00)
  F1      0.13 - 0.79   (SI: 0.13-0.79)


In [8]:
# Table S2: other-days VLM performance.
# Writes tables/vlm_simplified_performance_all_days.tex, reproducing the
# committed table EXACTLY (label tab:other-days-performance, which 05_SI.tex
# references): Date | Location | PPV | FOR, decimal CIs in brackets.
DAY_META = {
    "sep29":      ("9/29/23", "NYC"),
    "dec18":      ("12/18/23", "NYC"),
    "jan10":      ("1/10/24", "NYC"),
    "california": ("2/10/24", "SF Bay"),
}
DAY_ORDER = ["sep29", "dec18", "jan10", "california"]
# Combined labels retained for downstream cells (e.g. the alternate table).
DAY_LABELS = {d: f"{dt}, {lo}" for d, (dt, lo) in DAY_META.items()}

data_source = rows if (DO_BOOTSTRAP and 'rows' in locals()) else results
df_base = pd.DataFrame(data_source).set_index("day")


def _ci(row, key):
    v = row[key]
    lo = row.get(f"{key}_ci_lower")
    hi = row.get(f"{key}_ci_upper")
    if lo is None or pd.isna(lo) or hi is None or pd.isna(hi):
        return f"{v:.3f}"
    return f"{v:.3f} [{lo:.3f}, {hi:.3f}]"


body = []
for day in DAY_ORDER:
    r = df_base.loc[day]
    date, loc = DAY_META[day]
    ppv = _ci(r, "ppv_precision")
    forr = _ci(r, "false_omission_rate")
    body.append(f"{date} & {loc} & {ppv} & {forr} \\\\")

caption = (
    "Validation of VLM performance across multiple days and locations. We report "
    "the positive predictive value, $p(y=1|\\hat y = 1)$, and false omission rate, "
    "$p(y=1|\\hat y = 0)$. Classified positives ($\\hat y = 1$) are much likelier to "
    "show flooding ($y = 1$) than classified negatives across all four days. Results "
    "reported are for our preferred model (Cambrian-1-13B); 95\\% bootstrap confidence "
    "intervals (100,000 samples) are provided in brackets."
)

latex_code_simple = (
    "\\begin{table}[h!]\n\\centering\n\\small\n\\begin{tabular}{llcc}\n\\toprule\n"
    "Date & Location & $p(y=1|\\hat y = 1)$ & $p(y=1|\\hat y = 0)$ \\\\\n\\midrule\n"
    + "\n".join(body)
    + "\n\\bottomrule\n\\end{tabular}\n"
    + f"\\caption{{{caption}}}\n"
    + "\\label{tab:other-days-performance}\n\\end{table}\n"
)

print(latex_code_simple)

with open(PAPER_PATH / "tables" / "vlm_simplified_performance_all_days.tex", "w") as f:
    f.write(latex_code_simple)


\begin{table}[h!]
\centering
\small
\begin{tabular}{llcc}
\toprule
Date & Location & $p(y=1|\hat y = 1)$ & $p(y=1|\hat y = 0)$ \\
\midrule
9/29/23 & NYC & 0.658 [0.616, 0.700] & 0.006 [0.000, 0.014] \\
12/18/23 & NYC & 0.702 [0.606, 0.787] & 0.000 [0.000, 0.000] \\
1/10/24 & NYC & 0.812 [0.625, 1.000] & 0.000 [0.000, 0.000] \\
2/10/24 & SF Bay & 0.143 [0.000, 0.429] & 0.000 [0.000, 0.000] \\
\bottomrule
\end{tabular}
\caption{Validation of VLM performance across multiple days and locations. We report the positive predictive value, $p(y=1|\hat y = 1)$, and false omission rate, $p(y=1|\hat y = 0)$. Classified positives ($\hat y = 1$) are much likelier to show flooding ($y = 1$) than classified negatives across all four days. Results reported are for our preferred model (Cambrian-1-13B); 95\% bootstrap confidence intervals (100,000 samples) are provided in brackets.}
\label{tab:other-days-performance}
\end{table}



In [9]:
# Alternate Version: Raw Sample Metrics with Population-Adjusted Prevalence and Scientific Notation
# This follows the logic in bootstrap_matt_recall_f1.ipynb:
# p(y=1) = p(yhat=1)*p(y=1|yhat=1) + p(yhat=0)*p(y=1|yhat=0)

rng = np.random.default_rng(RANDOM_SEED)  # reproducible bootstrap

def raw_metrics_bootstrap_pop(tp, fp, tn, fn, p_yhat_1, n_boot=10000) -> dict[str, dict]:
    """Compute metrics with population-adjusted prevalence and bootstrap CIs."""
    n_pos_pred = tp + fp
    n_neg_pred = tn + fn
    
    # Observed rates (stratum-specific)
    p_ppv_obs = tp / n_pos_pred if n_pos_pred > 0 else 0
    p_for_obs = fn / n_neg_pred if n_neg_pred > 0 else 0
    
    # Observed population prevalence
    p_prev_obs = p_yhat_1 * p_ppv_obs + (1 - p_yhat_1) * p_for_obs
    
    # Bootstrap arrays
    boot_prev = []
    boot_ppv = []
    boot_for = []
    
    for _ in range(n_boot):
        # Sample stratum-specific rates from Binomial distributions
        s_ppv = rng.binomial(n_pos_pred, p_ppv_obs) / n_pos_pred if n_pos_pred > 0 else 0
        s_for = rng.binomial(n_neg_pred, p_for_obs) / n_neg_pred if n_neg_pred > 0 else 0
        
        # Compute population prevalence for this bootstrap sample
        s_prev = p_yhat_1 * s_ppv + (1 - p_yhat_1) * s_for
        
        boot_prev.append(s_prev)
        boot_ppv.append(s_ppv)
        boot_for.append(s_for)
        
    def get_stats(arr, val_obs):
        return {
            "val": val_obs,
            "low": np.percentile(arr, 2.5),
            "high": np.percentile(arr, 97.5)
        }

    return {
        "prevalence": get_stats(boot_prev, p_prev_obs),
        "ppv": get_stats(boot_ppv, p_ppv_obs),
        "for_val": get_stats(boot_for, p_for_obs),
    }

raw_rows = []
for day in ["sep29", "dec18", "jan10", "california"]:
    if day == "sep29":
        gt = load_sep29_gt()
        paths = sep29_chunk_paths()
    else:
        gt = load_otherday_gt(day, ANNOTATED_SAMPLE_PATHS[day])
        paths = CHUNK_PATHS[day]
        
    wanted = set(gt["image_name"].unique())
    preds = load_chunk_preds(paths, wanted)
    merged = gt.merge(preds, on="image_name", how="left").dropna(subset=["pred_bin"]).copy()
    merged["pred_bin"] = merged["pred_bin"].astype(int)
    
    y_true = merged["gt_bin"].to_numpy(dtype=int)
    y_pred = merged["pred_bin"].to_numpy(dtype=int)
    
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    
    # Get population predicted positive rate from POPULATION_STATS
    pop_stats = POPULATION_STATS[day]
    p_yhat_1 = pop_stats["pos"] / pop_stats["N"]
    
    day_label = DAY_LABELS[day]
    date_str, loc_str = day_label.split(", ")
    
    m = raw_metrics_bootstrap_pop(tp, fp, tn, fn, p_yhat_1)
    raw_rows.append({
        "Date": date_str,
        "Location": loc_str,
        "prev": m["prevalence"],
        "ppv": m["ppv"],
        "for_val": m["for_val"]
    })

caption = "Validation of VLM performance across multiple days and locations. Results reported are for our preferred model (Cambrian-1-13B). Classified positives ($\\hat y = 1$) are much likelier to show flooding ($y = 1$) than classified negatives across all four days. Prevalence $p(y=1)$ is estimated using population-adjusted weights. Metrics include 95% bootstrap confidence intervals computed from raw sample counts."
label = "tab:other-days-performance"

def fmt_sci(d):
    if pd.isna(d["val"]): return "--"
    def _fmt(v):
        if v == 0: return "0.00"
        s = "{:.2e}".format(v)
        b, e = s.split("e")
        return "{} \\cdot 10^{{{}}}".format(b, int(e))
    return "${} \\ [ {}, {} ]$".format(_fmt(d["val"]), _fmt(d["low"]), _fmt(d["high"]))

def fmt_dec(d):
    if pd.isna(d["val"]): return "--"
    return "{:.3f} [{:.3f}, {:.3f}]".format(d["val"], d["low"], d["high"])

latex_lines = [
    "\\begin{table}[h!]",
    "\\centering",
    "\\small",
    "\\begin{tabular}{llccc}",
    "\\toprule",
    "Date & Location & $p(y=1)$ & $p(y=1|\\hat y = 1)$ & $p(y=1|\\hat y = 0)$ \\\\",
    "\\midrule"
]

for r in raw_rows:
    row_str = "{} & {} & {} & {} & {} \\\\".format(
        r["Date"], r["Location"], fmt_sci(r["prev"]), fmt_dec(r["ppv"]), fmt_dec(r["for_val"])
    )
    latex_lines.append(row_str)

latex_lines.extend([
    "\\bottomrule",
    "\\end{tabular}",
    "\\caption{" + caption + "}",
    "\\label{" + label + "}",
    "\\end{table}"
])

latex_raw = "\n".join(latex_lines)
print("% Raw Metrics LaTeX Table with Scientific Notation for Prevalence")
print(latex_raw)


% Raw Metrics LaTeX Table with Scientific Notation for Prevalence
\begin{table}[h!]
\centering
\small
\begin{tabular}{llccc}
\toprule
Date & Location & $p(y=1)$ & $p(y=1|\hat y = 1)$ & $p(y=1|\hat y = 0)$ \\
\midrule
9/29/23 & NYC & $7.03 \cdot 10^{-3} \ [ 1.04 \cdot 10^{-3}, 1.50 \cdot 10^{-2} ]$ & 0.658 [0.616, 0.700] & 0.006 [0.000, 0.014] \\
12/18/23 & NYC & $3.32 \cdot 10^{-4} \ [ 2.87 \cdot 10^{-4}, 3.73 \cdot 10^{-4} ]$ & 0.702 [0.606, 0.787] & 0.000 [0.000, 0.000] \\
1/10/24 & NYC & $8.11 \cdot 10^{-5} \ [ 6.24 \cdot 10^{-5}, 9.99 \cdot 10^{-5} ]$ & 0.812 [0.625, 1.000] & 0.000 [0.000, 0.000] \\
2/10/24 & SF Bay & $4.17 \cdot 10^{-5} \ [ 0.00, 1.25 \cdot 10^{-4} ]$ & 0.143 [0.000, 0.429] & 0.000 [0.000, 0.000] \\
\bottomrule
\end{tabular}
\caption{Validation of VLM performance across multiple days and locations. Results reported are for our preferred model (Cambrian-1-13B). Classified positives ($\hat y = 1$) are much likelier to show flooding ($y = 1$) than classified negative